# 04 — Job Clustering & Topic Discovery

Discover natural career domain clusters across job postings using:
- **KMeans Clustering** on TF-IDF vectors
- **Elbow Method & Silhouette Scores** for evaluating optimal $k$
- **PCA 2D Dimensionality Reduction** for cluster visualization
- **NMF Topic Modeling** for latent skill theme extraction

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.models.recommender import load_recommender
from src.models.clustering import fit_job_clusters, evaluate_elbow_silhouette, fit_nmf_topics

# Load precalculated job catalog and sparse TF-IDF matrix
job_vec, job_matrix, job_meta = load_recommender()
print(f"Catalog size: {job_matrix.shape[0]:,} jobs | Features: {job_matrix.shape[1]:,}")

## 1. Elbow Method & Silhouette Evaluation

In [ ]:
# Sample 3,000 jobs for evaluation
sample_matrix = job_matrix[:3000]
k_values = [3, 5, 8, 10]
eval_results = evaluate_elbow_silhouette(sample_matrix, k_range=k_values)

fig, ax1 = plt.subplots(1, 2, figsize=(14, 5))

# Inertia (Elbow)
ax1[0].plot(k_values, eval_results["inertias"], marker="o", color="#2563EB", linewidth=2)
ax1[0].set_title("Elbow Method (Inertia vs. k)")
ax1[0].set_xlabel("Number of Clusters (k)")
ax1[0].set_ylabel("Inertia")
ax1[0].grid(True, linestyle="--", alpha=0.6)

# Silhouette Scores
ax1[1].plot(k_values, eval_results["silhouette_scores"], marker="s", color="#10B981", linewidth=2)
ax1[1].set_title("Silhouette Score vs. k")
ax1[1].set_xlabel("Number of Clusters (k)")
ax1[1].set_ylabel("Silhouette Score")
ax1[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 2. Fit KMeans (k=6) & Extract Domain Keywords

In [ ]:
cluster_res = fit_job_clusters(sample_matrix, job_vec, n_clusters=6)

print("=== Cluster Keyword Themes ===")
for cid, terms in cluster_res["cluster_topics"].items():
    print(f"Cluster {cid}: {', '.join(terms[:8])}")

## 3. PCA 2D Cluster Visualization

In [ ]:
# Reduce TF-IDF dimensions to 2D using PCA
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(sample_matrix.toarray())
labels = cluster_res["labels"]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=coords_2d[:, 0],
    y=coords_2d[:, 1],
    hue=labels,
    palette="tab10",
    alpha=0.6,
    s=30,
)
plt.title("PCA 2D Projection of Job Clusters", fontsize=13)
plt.xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(title="Cluster ID", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. NMF Topic Modeling (Latent Skill Themes)

In [ ]:
topics = fit_nmf_topics(sample_matrix, job_vec, n_topics=6, top_words=8)
print("=== NMF Latent Topics ===")
for tid, words in topics.items():
    print(f"Topic #{tid+1}: {', '.join(words)}")